# PACE OCI Granule Preprocessing

This notebook processes PACE OCI L1B science data by:
1. Searching for granules within a specified time range (September 2-4, 2025)
2. Extracting random scan lines from each granule
3. Combining rhot_blue, rhot_red, and rhot_SWIR data arrays
4. Saving the processed data to S3 as Parquet files

The workflow authenticates with both NASA Earthdata (for data access) and AWS (for S3 storage).

## 1. Imports and Authentication

In [ ]:
from time import sleep
import os
import random
import re


import pandas as pd
import xarray as xr
from xarray.backends.api import open_datatree
import numpy as np
import boto3
import earthaccess


from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Authenticate with NASA Earthdata.
# Uses EARTHDATA_USERNAME / EARTHDATA_PASSWORD from .env (loaded above);
# falls back to an interactive prompt if those aren't set.
if os.getenv("EARTHDATA_USERNAME") and os.getenv("EARTHDATA_PASSWORD"):
    earthaccess.login(strategy="environment")
else:
    earthaccess.login()

In [ ]:
boto3.setup_default_session(
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
)

# 2. Sample Data

Extract granuls sampled scans from September 2nd - September 4th

In [ ]:
tspan = ('2025-09-02T00:00:00Z', '2025-09-04T00:00:00Z')

results = earthaccess.search_data(
    short_name = "PACE_OCI_L1B_SCI",
    temporal=tspan,
    count=10000
)

granules = list([re.search(r"(\d{8}T\d{6})", res.dataviz_links()[0]).group(1) for res in results])

In [ ]:
bucket = os.getenv("BUCKET_NAME")

def process_granule(granule_id):
    results = earthaccess.search_data(
    short_name="PACE_OCI_L1B_SCI",
    granule_name=f"PACE_OCI.{granule_id}.L1B.V3.nc",)

    random_scan = random.randint(0, 1065)

    print(f"Processing granule {granule_id} with random scan {random_scan}")

    paths = earthaccess.open(results)
    datatree = open_datatree(paths[0])
    dataset = xr.merge(datatree.to_dict().values())

    rhot_blue_arr = dataset["rhot_blue"].isel(scans=random_scan).values
    rhot_red_arr = dataset["rhot_red"].isel(scans=random_scan).values
    rhot_SWIR_arr = dataset["rhot_SWIR"].isel(scans=random_scan).values
    tostore = np.vstack([rhot_blue_arr, rhot_red_arr, rhot_SWIR_arr]).T

    path = f"s3://{bucket}/samples/{granule_id}/scan{random_scan}.parquet"

    df = pd.DataFrame(tostore)
    df.to_parquet(
        path,
        index=False,
        engine="pyarrow",
    )

    return path

In [ ]:
for granule_id in granules:
    print(f"Scheduling processing for granule {granule_id}")
    process_granule(granule_id)
    sleep(60)  # To avoid overwhelming the server
    print(f"Completed processing for granule {granule_id}")